# SMS Spam Detection Using Natural Language Processing

---

| Field | Details |
|---|---|
| **Student Name** | Rabeea Anjum |
| **Course** | Natural Language Processing |
| **Department** | Computer Systems Engineering |
| **University** | The Islamia University of Bahawalpur |
| **Submitted To** | Dr. Nadia Rasheed |

---

## Project Overview
This notebook implements a complete **NLP pipeline** for binary SMS spam classification.

### Pipeline Stages
```
Data Loading → Preprocessing → TF-IDF Feature Extraction → Label Encoding
    → Train-Test Split → Model Training → Evaluation → Sample Predictions
```

---
## Cell 1 — Install Required Libraries

In [ ]:
# Install required libraries (run once in Google Colab)
!pip install nltk scikit-learn pandas numpy matplotlib seaborn

---
## Cell 2 — Import Libraries

In [ ]:
# ── Standard libraries ──────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
import warnings
warnings.filterwarnings('ignore')

# ── NLP libraries ───────────────────────────────────────────────────────────
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

# ── Machine Learning libraries ──────────────────────────────────────────────
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

print("✅ All libraries imported successfully.")

---
## Cell 3 — Download NLTK Data

In [ ]:
# Download NLTK stopwords corpus
nltk.download('stopwords')
print("✅ NLTK stopwords downloaded.")

---
## Cell 4 — Upload Dataset

> Upload the `spam.csv` file when prompted.  
> Dataset source: https://archive.ics.uci.edu/ml/datasets/sms+spam+collection

In [ ]:
# Upload dataset in Google Colab
from google.colab import files

print("Please upload the spam.csv file:")
uploaded = files.upload()

---
## Cell 5 — Load Dataset

In [ ]:
# Load dataset into a Pandas DataFrame
df = pd.read_csv('spam.csv', encoding='latin-1')

# Keep only relevant columns and rename
df = df[['v1', 'v2']]
df.columns = ['label', 'message']

print("Dataset Shape:", df.shape)
print("\nFirst 5 Records:")
display(df.head())

---
## Cell 6 — Dataset Information & Class Distribution

In [ ]:
# Dataset structure and class balance
print("Dataset Info:")
print(df.info())

print("\nClass Distribution:")
print(df['label'].value_counts())
print("\nClass Proportions:")
print(df['label'].value_counts(normalize=True).round(3))

# Visualize class distribution
plt.figure(figsize=(6, 4))
df['label'].value_counts().plot(kind='bar', color=["#2196F3", "#F44336"], edgecolor='black')
plt.title('Class Distribution: Ham vs Spam', fontsize=13, fontweight='bold')
plt.xlabel('Label')
plt.ylabel('Count')
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig('class_distribution.png', dpi=150)
plt.show()
print("✅ Class distribution plot saved.")

---
## Cell 7 — Text Preprocessing Function

### NLP Preprocessing Pipeline:
1. **Lowercase Conversion** — normalize case  
2. **Remove Special Characters** — remove punctuation, digits, symbols  
3. **Tokenization** — split text into individual words  
4. **Stop-word Removal** — remove high-frequency meaningless words  
5. **Porter Stemming** — reduce words to root forms  

In [ ]:
# Initialize Porter Stemmer
ps = PorterStemmer()

def preprocess(text):
    """
    Apply full NLP preprocessing pipeline to a single SMS message.

    Steps:
        1. Lowercase conversion
        2. Remove special characters and digits (keep only letters)
        3. Tokenize by splitting on whitespace
        4. Remove English stop-words
        5. Apply Porter Stemming to each remaining token

    Args:
        text (str): Raw SMS message string.

    Returns:
        str: Cleaned, preprocessed text string.
    """
    # Step 1: Lowercase
    text = text.lower()

    # Step 2: Remove special characters, digits, punctuation
    text = re.sub('[^a-zA-Z]', ' ', text)

    # Step 3 + 4 + 5: Tokenize, remove stopwords, apply stemming
    words = [
        ps.stem(word)
        for word in text.split()
        if word not in stopwords.words('english')
    ]

    return ' '.join(words)

print("✅ Preprocessing function defined.")

# Quick test
test_msg = "FREE OFFER!! Call NOW to claim your prize worth $1000."
print("\nExample:")
print("  Input : ", test_msg)
print("  Output:", preprocess(test_msg))

---
## Cell 8 — Apply Preprocessing to Entire Dataset

In [ ]:
# Apply preprocessing pipeline to all messages
df['clean_text'] = df['message'].apply(preprocess)

print("✅ Preprocessing applied to all", len(df), "messages.")
print("\nBefore vs After Preprocessing (sample):")
display(df[['message', 'clean_text']].head(8))

---
## Cell 9 — TF-IDF Feature Extraction

**TF-IDF** (Term Frequency – Inverse Document Frequency) converts text into numerical feature vectors.  
- `max_features=5000` → vocabulary limited to 5,000 most important terms.

In [ ]:
# Initialize TF-IDF Vectorizer with 5000 maximum features
tfidf = TfidfVectorizer(max_features=5000)

# Fit and transform the cleaned text corpus
X = tfidf.fit_transform(df['clean_text']).toarray()

print("✅ TF-IDF Vectorization complete.")
print("Feature Matrix Shape:", X.shape)
print("  → Rows : Number of SMS messages")
print("  → Cols : Number of TF-IDF features (vocabulary size)")

---
## Cell 10 — Label Encoding

In [ ]:
# Encode categorical labels to numeric values
# ham  → 0 (legitimate message)
# spam → 1 (spam message)
y = df['label'].map({'ham': 0, 'spam': 1})

print("✅ Labels encoded:  ham=0,  spam=1")
print("\nEncoded Label Distribution:")
print(y.value_counts())

---
## Cell 11 — Train-Test Split

- **80%** Training Set → model learning  
- **20%** Testing Set → model evaluation  
- `random_state=42` → reproducibility

In [ ]:
# Split dataset: 80% train, 20% test
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

print("✅ Train-Test Split Complete")
print(f"  Training Samples : {len(X_train)} ({len(X_train)/len(X)*100:.1f}%)")
print(f"  Testing  Samples : {len(X_test)}  ({len(X_test)/len(X)*100:.1f}%)")

---
## Cell 12 — Train Multinomial Naive Bayes Model

In [ ]:
# Initialize and train Multinomial Naive Bayes classifier
model = MultinomialNB()
model.fit(X_train, y_train)

print("✅ Multinomial Naive Bayes model trained successfully.")
print("   Model Type  :", type(model).__name__)
print("   Training Size:", len(X_train), "samples")

---
## Cell 13 — Make Predictions

In [ ]:
# Generate predictions on test set
y_pred = model.predict(X_test)

print("✅ Predictions generated for", len(y_pred), "test samples.")
print("   Predicted Spam Count:", sum(y_pred == 1))
print("   Predicted Ham  Count:", sum(y_pred == 0))

---
## Cell 14 — Accuracy Score

In [ ]:
# Compute overall accuracy
accuracy = accuracy_score(y_test, y_pred)

print("=" * 45)
print(f"  Overall Accuracy : {accuracy * 100:.2f}%")
print("=" * 45)

---
## Cell 15 — Classification Report

Detailed per-class metrics: **Precision**, **Recall**, **F1-Score**, **Support**

In [ ]:
# Print full classification report
print("Classification Report")
print("=" * 55)
print(classification_report(y_test, y_pred, target_names=['Ham (0)', 'Spam (1)']))

---
## Cell 16 — Confusion Matrix

In [ ]:
# Compute and visualize confusion matrix
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(6, 5))
sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=['Ham (0)', 'Spam (1)'],
    yticklabels=['Ham (0)', 'Spam (1)'],
    linewidths=0.5,
    linecolor='gray'
)
plt.title('Confusion Matrix – SMS Spam Detection', fontsize=13, fontweight='bold', pad=12)
plt.xlabel('Predicted Label', fontsize=11)
plt.ylabel('Actual Label', fontsize=11)
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150)
plt.show()

print("✅ Confusion matrix saved as confusion_matrix.png")
print(f"\nTrue  Negatives (TN – Ham  ✓) : {cm[0][0]}")
print(f"False Positives (FP – Ham  ✗) : {cm[0][1]}")
print(f"False Negatives (FN – Spam ✗) : {cm[1][0]}")
print(f"True  Positives (TP – Spam ✓) : {cm[1][1]}")

---
## Cell 17 — Define Sample Test Messages

In [ ]:
# Five custom SMS messages for real-world testing
samples = [
    "Congratulations! You won a free iPhone. Click now to claim.",   # → Spam
    "Can we meet tomorrow for the project discussion?",               # → Ham
    "Claim your lottery cash prize of $5000 immediately.",           # → Spam
    "Please submit the assignment before Friday.",                   # → Ham
    "Win free rewards by clicking this link. Limited offer!",        # → Spam
]

print("✅ Sample messages defined.")
for i, msg in enumerate(samples, 1):
    print(f"  {i}. {msg}")

---
## Cell 18 — Transform Sample Messages & Predict

In [ ]:
# Preprocess sample messages
samples_clean = [preprocess(msg) for msg in samples]

# Transform using the fitted TF-IDF vectorizer (do NOT re-fit)
sample_features = tfidf.transform(samples_clean)

# Predict labels and confidence probabilities
predictions = model.predict(sample_features)
probs       = model.predict_proba(sample_features)

print("✅ Sample messages transformed and predictions generated.")

---
## Cell 19 — Display Predictions with Confidence Scores

In [ ]:
print("\n" + "=" * 65)
print("          SAMPLE MESSAGE PREDICTIONS")
print("=" * 65)

for i, text in enumerate(samples):
    label      = "🔴 SPAM" if predictions[i] == 1 else "🟢 HAM"
    confidence = np.max(probs[i]) * 100

    print(f"\nMessage {i+1}:")
    print(f"  Text       : {text}")
    print(f"  Prediction : {label}")
    print(f"  Confidence : {confidence:.2f}%")
    print("-" * 65)

# Save predictions as a DataFrame
results_df = pd.DataFrame({
    'Message'   : samples,
    'Prediction': ['Spam' if p == 1 else 'Ham' for p in predictions],
    'Confidence': [f"{np.max(probs[i])*100:.2f}%" for i in range(len(samples))]
})

print("\nPredictions Summary Table:")
display(results_df)

---
## Cell 20 — Final Project Summary

In [ ]:
print("\n" + "=" * 55)
print("       PROJECT SUMMARY – SMS SPAM DETECTION")
print("=" * 55)
print(f"  Student Name      : Rabeea Anjum")
print(f"  Course            : Natural Language Processing")
print(f"  Model             : Multinomial Naive Bayes")
print(f"  Feature Extractor : TF-IDF (5000 features)")
print("-" * 55)
print(f"  Total Dataset     : {len(df)} messages")
print(f"  Training Samples  : {len(X_train)}")
print(f"  Testing  Samples  : {len(X_test)}")
print("-" * 55)
print(f"  Final Accuracy    : {accuracy * 100:.2f}%")
print("=" * 55)
print("\n✅ SMS Spam Detection Project Completed Successfully!")